# 01 - Exploratory Data Analysis

This notebook does a quick EDA over the Scenario-01 SaaS dataset (customers, products, subscriptions, payments).

It reads the CSVs in `data/raw/`, so it runs with pandas alone and needs no database. The CSVs are regenerated from `docker/mysql/init.sql` by `scripts/export_csvs.py`, so they match the MySQL practice environment.

An optional cell at the end shows how to run the same revenue aggregation against the Dockerized MySQL instance when it is available.

In [1]:
from pathlib import Path

import pandas as pd

# Anchor paths to the repo root so the notebook runs from any working directory
# (locally the cwd is notebooks/, inside the Docker image it is /app).
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO_ROOT / "data" / "raw"
OUTPUT = REPO_ROOT / "data" / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(RAW / "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(RAW / "products.csv")
subscriptions = pd.read_csv(
    RAW / "subscriptions.csv", parse_dates=["start_date", "end_date"]
)
payments = pd.read_csv(RAW / "payments.csv", parse_dates=["payment_date"])

print(
    f"customers={len(customers)} products={len(products)} "
    f"subscriptions={len(subscriptions)} payments={len(payments)}"
)

customers=30 products=12 subscriptions=60 payments=90


## Data quality checks

Confirm there are no orphan foreign keys and that the only nulls are the expected `end_date` on active subscriptions.

In [2]:
orphan_subs = set(subscriptions.customer_id) - set(customers.customer_id)
orphan_pay = set(payments.subscription_id) - set(subscriptions.subscription_id)
orphan_prod = set(subscriptions.product_id) - set(products.product_id)

print("orphan subscription.customer_id:", orphan_subs or "none")
print("orphan payment.subscription_id:", orphan_pay or "none")
print("orphan subscription.product_id:", orphan_prod or "none")
print()
print("null counts per column:")
print(subscriptions.isna().sum())

orphan subscription.customer_id: none
orphan payment.subscription_id: none
orphan subscription.product_id: none

null counts per column:
subscription_id     0
customer_id         0
product_id          0
start_date          0
end_date           46
status              0
dtype: int64


## Revenue by product

Join payments to subscriptions to products and sum the amount per product.

In [3]:
revenue_by_product = (
    payments.merge(subscriptions, on="subscription_id", how="left")
    .merge(products, on="product_id", how="left")
    .groupby("product_name", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "total_revenue"})
    .sort_values("total_revenue", ascending=False)
    .reset_index(drop=True)
)
revenue_by_product

,product_name,total_revenue
0,Supply Chain Opt,4614.0
1,Fraud Detector,4389.0
2,Basic Analytics,1972.0
3,Pro Analytics,1855.0
4,Enterprise Analytics,1797.0
5,Inventory Optimizer,747.0
6,Compliance Monitor,698.0
7,Marketing Insights,597.0
8,Credit Scoring,558.0
9,Logistics Planner,537.0


## Revenue and paying customers by country

In [4]:
by_country = (
    payments.merge(subscriptions, on="subscription_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .groupby("country")
    .agg(
        total_revenue=("amount", "sum"),
        paying_customers=("customer_id", "nunique"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
    .reset_index(drop=True)
)
by_country

,country,total_revenue,paying_customers
0,CA,8993.0,5
1,US,6252.0,3
2,BR,3255.0,2


## Write the result to data/output/

`data/output/` is gitignored, so this is a local artifact, not committed.

In [5]:
out_path = OUTPUT / "results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    revenue_by_product.to_excel(writer, sheet_name="revenue_by_product", index=False)
    by_country.to_excel(writer, sheet_name="revenue_by_country", index=False)
print("wrote", out_path)

wrote C:\Users\caiof\AppData\Local\Temp\gh-exec\MyCapitalOneTraining\data\output\results.xlsx


## Optional: same query against the Dockerized MySQL instance

This cell only runs when the DB env vars are set (they are, inside the `notebook` service in `docker-compose.yml`). Locally without the database it is skipped, so the notebook still runs end to end.

In [6]:
import os

db_host = os.getenv("DB_HOST")
if db_host:
    from sqlalchemy import create_engine

    engine = create_engine(
        f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
        f"@{db_host}/{os.getenv('DB_NAME', 'Scenario01')}"
    )
    df_sql = pd.read_sql(
        """
        SELECT p.product_name, SUM(pay.amount) AS total_revenue
        FROM payments pay
        JOIN subscriptions s ON pay.subscription_id = s.subscription_id
        JOIN products p ON s.product_id = p.product_id
        GROUP BY p.product_name
        ORDER BY total_revenue DESC
        """,
        engine,
    )
    display(df_sql)
else:
    print("DB_HOST not set; skipping the MySQL path (CSV results above are equivalent).")

DB_HOST not set; skipping the MySQL path (CSV results above are equivalent).
